## EoMT model fine-tuning
---

Here we show the steps and workflow of the fine-tuning process. We have tried to fine-tune two layers overall: head and the last two attention layers in 11th and 10th blocks.

Head layer, in its turn, is subdivided into sublayers and we fine-tuned and evaluated each of them separately to be able to see the training process and possible improvement.

While fine-tuning the attention blocks, the lower Learning Rate was used.

In [ ]:
from IPython.display import clear_output
!pip install -r /content/outlierdrive/eomt/requirements.txt
clear_output()

In [7]:
!pip install --upgrade wandb
clear_output()

In [6]:
import torch
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))

True
Tesla T4


## Head only fine-tuning

Notice which config file and checkpoint files we used: Cityscapes semantic config and COCO checkpoint, because we
fine-tune COCO model on Cityscapes.

Firstly, we try fine-tuning on one batch.

In [9]:
%cd /content/outlierdrive/eomt

/content/outlierdrive/eomt


Fine-tune COCO-trained model on cityscapes, head-only

In [ ]:
!python main.py fit \
    -c configs/dinov2/cityscapes/semantic/eomt_base_640.yaml \
    --data.path /content/drive/MyDrive/cityscapes \
    --model.ckpt_path /content/eomt_coco.bin \
    --model.load_ckpt_class_head False \
    --model.network.num_q 200 \
    --data.img_size "[640,640]" \
    --trainer.devices 1 \
    --trainer.max_epochs 10 \
    --trainer.limit_val_batches 0 \
    --data.batch_size 16 \
    --data.num_workers 2 \
    --trainer.callbacks+=lightning.pytorch.callbacks.ModelCheckpoint \
    --trainer.callbacks.dirpath /content/drive/MyDrive/eomt_finetuning/head_only_all_epochs \
    --trainer.callbacks.filename "epoch={epoch}-step={step}" \
    --trainer.callbacks.save_top_k -1 \
    --trainer.callbacks.every_n_epochs 1 \
    --trainer.callbacks.save_last True

2026-06-03 13:42:29.423099: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-06-03 13:42:29.487128: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
Seed set to 0
Trainable parameters:
class_head.weight
class_head.bias
mask_head.0.weight
mask_head.0.bias
mask_head.2.weight
mask_head.2.bias
mask_head.4.weight
mask_head.4.bias
INFO:root:Loaded 195 keys
Using 16bit Automatic Mixed Precision (AMP)
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
wandb: WAR

Evaluate head-only finetuned model on my script on all 19 classes

Here we evaluate on a checkpoints after 5th epoch on head-only fine-tuning

In [10]:
!python /content/outlierdrive/step4_eomt_eval/eomt_eval_iou.py \
  --repo-root /content/outlierdrive/eomt \
  --config /content/outlierdrive/eomt/configs/dinov2/cityscapes/semantic/eomt_base_640.yaml \
  --checkpoint /content/drive/MyDrive/eomt_finetuning/head_only_all_epochs/epoch=epoch=4-step=step=925.ckpt \
  --data-path /content/drive/MyDrive/cityscapes \
  --output-dir /content/drive/MyDrive/eomt_valset_predictions/coco_finetuned_head_only/5_epochs \
  --device cuda:0 \
  --img-size 640 640 \
  --num-q 200 \
  --num-workers 2

Repo root: /content/outlierdrive/eomt
Config: /content/outlierdrive/eomt/configs/dinov2/cityscapes/semantic/eomt_base_640.yaml
Checkpoint: /content/drive/MyDrive/eomt_finetuning/head_only_all_epochs/epoch=epoch=4-step=step=925.ckpt
Data path: /content/drive/MyDrive/cityscapes
Device: cuda:0
Image size: (640, 640)
num_q: 200
Data module: <class 'datasets.cityscapes_semantic.CityscapesSemantic'>
Data img_size: (640, 640)
Data num_classes: 19
model.safetensors: 100% 346M/346M [00:03<00:00, 104MB/s]
2026-06-09 09:18:09.250383: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
Trainable parameters:
encoder.backbone.blocks.10.attn.qkv.weight
encoder.backbone.blocks.10.attn.qkv.bias
encoder.backbone.blocks.10.attn.proj.weight
encoder.backbone.blocks.10.at

Here we evaluate on a checkpoints after 10th epoch on head-only fine-tuning

In [ ]:
!python /content/outlierdrive/step4_eomt_eval/eomt_eval_iou.py \
  --repo-root /content/outlierdrive/eomt \
  --config /content/outlierdrive/eomt/configs/dinov2/cityscapes/semantic/eomt_base_640.yaml \
  --checkpoint /content/drive/MyDrive/eomt_finetuning/head_only_all_epochs/epoch=epoch=9-step=step=1850.ckpt \
  --data-path /content/drive/MyDrive/cityscapes \
  --output-dir /content/drive/MyDrive/eomt_valset_predictions/coco_finetuned_head_only \
  --device cuda:0 \
  --img-size 640 640 \
  --num-q 200 \
  --num-workers 2

Repo root: /content/outlierdrive/eomt
Config: /content/outlierdrive/eomt/configs/dinov2/cityscapes/semantic/eomt_base_640.yaml
Checkpoint: /content/drive/MyDrive/eomt_finetuning/head_only_all_epochs/epoch=epoch=9-step=step=1850.ckpt
Data path: /content/drive/MyDrive/cityscapes
Device: cuda:0
Image size: (640, 640)
num_q: 200
Data module: <class 'datasets.cityscapes_semantic.CityscapesSemantic'>
Data img_size: (640, 640)
Data num_classes: 19
model.safetensors: 100% 346M/346M [04:32<00:00, 1.27MB/s]
2026-06-03 17:03:13.558526: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-06-03 17:03:13.620387: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the fo

## Head and upscale finetuning

In [ ]:
!python main.py fit \
    -c configs/dinov2/cityscapes/semantic/eomt_base_640.yaml \
    --data.path /content/drive/MyDrive/cityscapes \
    --model.ckpt_path /content/eomt_coco.bin \
    --model.load_ckpt_class_head False \
    --model.network.num_q 200 \
    --data.img_size "[640,640]" \
    --trainer.devices 1 \
    --trainer.max_epochs 10 \
    --trainer.limit_val_batches 0 \
    --data.batch_size 16 \
    --data.num_workers 2 \
    --trainer.callbacks+=lightning.pytorch.callbacks.ModelCheckpoint \
    --trainer.callbacks.dirpath /content/drive/MyDrive/eomt_finetuning/head_and_upscaling \
    --trainer.callbacks.filename "epoch={epoch}-step={step}" \
    --trainer.callbacks.save_top_k -1 \
    --trainer.callbacks.every_n_epochs 1 \
    --trainer.callbacks.save_last True

2026-06-03 20:30:49.121545: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-06-03 20:30:49.186697: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
Seed set to 0
Trainable parameters:
class_head.weight
class_head.bias
mask_head.0.weight
mask_head.0.bias
mask_head.2.weight
mask_head.2.bias
mask_head.4.weight
mask_head.4.bias
upscale.0.conv1.weight
upscale.0.conv1.bias
upscale.0.conv2.weight
upscale.0.norm.weight
upscale.0.norm.bias
upscale.1.conv1.weight
upscale.1.conv1.bias
upscale.1.conv2.weight
upscale.1.norm.weigh

Evaluate the head-and-upscale fine-tuned model, on the checkpoints of 5th epoch.

In [12]:
!python /content/outlierdrive/step4_eomt_eval/eomt_eval_iou.py \
  --repo-root /content/outlierdrive/eomt \
  --config /content/outlierdrive/eomt/configs/dinov2/cityscapes/semantic/eomt_base_640.yaml \
  --checkpoint /content/drive/MyDrive/eomt_finetuning/head_and_upscaling/epoch=epoch=4-step=step=925.ckpt \
  --data-path /content/drive/MyDrive/cityscapes \
  --output-dir /content/drive/MyDrive/eomt_valset_predictions/coco_finetuned_head_upscale/5_epochs \
  --device cuda:0 \
  --img-size 640 640 \
  --num-q 200 \
  --num-workers 2

Repo root: /content/outlierdrive/eomt
Config: /content/outlierdrive/eomt/configs/dinov2/cityscapes/semantic/eomt_base_640.yaml
Checkpoint: /content/drive/MyDrive/eomt_finetuning/head_and_upscaling/epoch=epoch=4-step=step=925.ckpt
Data path: /content/drive/MyDrive/cityscapes
Device: cuda:0
Image size: (640, 640)
num_q: 200
Data module: <class 'datasets.cityscapes_semantic.CityscapesSemantic'>
Data img_size: (640, 640)
Data num_classes: 19
2026-06-09 09:30:20.645650: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
Trainable parameters:
encoder.backbone.blocks.10.attn.qkv.weight
encoder.backbone.blocks.10.attn.qkv.bias
encoder.backbone.blocks.10.attn.proj.weight
encoder.backbone.blocks.10.attn.proj.bias
encoder.backbone.blocks.11.attn.qkv.weight
enc

Evaluate the head-and-upscale fine-tuned model, on the checkpoints of 10th epoch.

In [ ]:
!python /content/outlierdrive/step4_eomt_eval/eomt_eval_iou.py \
  --repo-root /content/outlierdrive/eomt \
  --config /content/outlierdrive/eomt/configs/dinov2/cityscapes/semantic/eomt_base_640.yaml \
  --checkpoint /content/drive/MyDrive/eomt_finetuning/head_and_upscaling/epoch=epoch=9-step=step=1850.ckpt \
  --data-path /content/drive/MyDrive/cityscapes \
  --output-dir /content/drive/MyDrive/eomt_valset_predictions/coco_finetuned_head_upscale \
  --device cuda:0 \
  --img-size 640 640 \
  --num-q 200 \
  --num-workers 2

Repo root: /content/outlierdrive/eomt
Config: /content/outlierdrive/eomt/configs/dinov2/cityscapes/semantic/eomt_base_640.yaml
Checkpoint: /content/drive/MyDrive/eomt_finetuning/head_and_upscaling/epoch=epoch=9-step=step=1850.ckpt
Data path: /content/drive/MyDrive/cityscapes
Device: cuda:0
Image size: (640, 640)
num_q: 200
Data module: <class 'datasets.cityscapes_semantic.CityscapesSemantic'>
Data img_size: (640, 640)
Data num_classes: 19
model.safetensors: 100% 346M/346M [00:01<00:00, 222MB/s]
2026-06-04 05:06:55.895942: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-06-04 05:06:55.959506: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the follo

## Head + upscale + q fine-tuning

In [ ]:
!python main.py fit \
    -c configs/dinov2/cityscapes/semantic/eomt_base_640.yaml \
    --data.path /content/drive/MyDrive/cityscapes \
    --model.ckpt_path /content/eomt_coco.bin \
    --model.load_ckpt_class_head False \
    --model.network.num_q 200 \
    --data.img_size "[640,640]" \
    --trainer.devices 1 \
    --trainer.max_epochs 10 \
    --trainer.limit_val_batches 0 \
    --data.batch_size 16 \
    --data.num_workers 2 \
    --trainer.callbacks+=lightning.pytorch.callbacks.ModelCheckpoint \
    --trainer.callbacks.dirpath /content/drive/MyDrive/eomt_finetuning/head_upscale_q \
    --trainer.callbacks.filename "epoch={epoch}-step={step}" \
    --trainer.callbacks.save_top_k -1 \
    --trainer.callbacks.every_n_epochs 1 \
    --trainer.callbacks.save_last True

2026-06-04 19:06:48.892961: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-06-04 19:06:48.955947: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
Seed set to 0
Trainable parameters:
q.weight
class_head.weight
class_head.bias
mask_head.0.weight
mask_head.0.bias
mask_head.2.weight
mask_head.2.bias
mask_head.4.weight
mask_head.4.bias
upscale.0.conv1.weight
upscale.0.conv1.bias
upscale.0.conv2.weight
upscale.0.norm.weight
upscale.0.norm.bias
upscale.1.conv1.weight
upscale.1.conv1.bias
upscale.1.conv2.weight
upscale.1.n

Evaluate head+upscale+q fine-tuned model on 5th epoch

In [13]:
!python /content/outlierdrive/step4_eomt_eval/eomt_eval_iou.py \
  --repo-root /content/outlierdrive/eomt \
  --config /content/outlierdrive/eomt/configs/dinov2/cityscapes/semantic/eomt_base_640.yaml \
  --checkpoint /content/drive/MyDrive/eomt_finetuning/head_upscale_q/epoch=epoch=4-step=step=925.ckpt \
  --data-path /content/drive/MyDrive/cityscapes \
  --output-dir /content/drive/MyDrive/eomt_valset_predictions/coco_finetuned_head_upsc_q/5_epoch \
  --device cuda:0 \
  --img-size 640 640 \
  --num-q 200 \
  --num-workers 2

Repo root: /content/outlierdrive/eomt
Config: /content/outlierdrive/eomt/configs/dinov2/cityscapes/semantic/eomt_base_640.yaml
Checkpoint: /content/drive/MyDrive/eomt_finetuning/head_upscale_q/epoch=epoch=4-step=step=925.ckpt
Data path: /content/drive/MyDrive/cityscapes
Device: cuda:0
Image size: (640, 640)
num_q: 200
Data module: <class 'datasets.cityscapes_semantic.CityscapesSemantic'>
Data img_size: (640, 640)
Data num_classes: 19
2026-06-09 09:39:02.028120: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
Trainable parameters:
encoder.backbone.blocks.10.attn.qkv.weight
encoder.backbone.blocks.10.attn.qkv.bias
encoder.backbone.blocks.10.attn.proj.weight
encoder.backbone.blocks.10.attn.proj.bias
encoder.backbone.blocks.11.attn.qkv.weight
encoder

Evaluate head+upscale+q fine-tuned model on 8th epoch

In [14]:
!python /content/outlierdrive/step4_eomt_eval/eomt_eval_iou.py \
  --repo-root /content/outlierdrive/eomt \
  --config /content/outlierdrive/eomt/configs/dinov2/cityscapes/semantic/eomt_base_640.yaml \
  --checkpoint /content/drive/MyDrive/eomt_finetuning/head_upscale_q/epoch=epoch=7-step=step=1480.ckpt \
  --data-path /content/drive/MyDrive/cityscapes \
  --output-dir /content/drive/MyDrive/eomt_valset_predictions/coco_finetuned_head_upsc_q/8_epoch \
  --device cuda:0 \
  --img-size 640 640 \
  --num-q 200 \
  --num-workers 2

Repo root: /content/outlierdrive/eomt
Config: /content/outlierdrive/eomt/configs/dinov2/cityscapes/semantic/eomt_base_640.yaml
Checkpoint: /content/drive/MyDrive/eomt_finetuning/head_upscale_q/epoch=epoch=7-step=step=1480.ckpt
Data path: /content/drive/MyDrive/cityscapes
Device: cuda:0
Image size: (640, 640)
num_q: 200
Data module: <class 'datasets.cityscapes_semantic.CityscapesSemantic'>
Data img_size: (640, 640)
Data num_classes: 19
2026-06-09 09:48:09.714595: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
Trainable parameters:
encoder.backbone.blocks.10.attn.qkv.weight
encoder.backbone.blocks.10.attn.qkv.bias
encoder.backbone.blocks.10.attn.proj.weight
encoder.backbone.blocks.10.attn.proj.bias
encoder.backbone.blocks.11.attn.qkv.weight
encode

Evaluate head+upscale+q fine-tuned model on 10th epoch

In [ ]:
!python /content/outlierdrive/step4_eomt_eval/eomt_eval_iou.py \
  --repo-root /content/outlierdrive/eomt \
  --config /content/outlierdrive/eomt/configs/dinov2/cityscapes/semantic/eomt_base_640.yaml \
  --checkpoint /content/drive/MyDrive/eomt_finetuning/head_upscale_q/epoch=epoch=9-step=step=1850.ckpt \
  --data-path /content/drive/MyDrive/cityscapes \
  --output-dir /content/drive/MyDrive/eomt_valset_predictions/coco_finetuned_head_upsc_q \
  --device cuda:0 \
  --img-size 640 640 \
  --num-q 200 \
  --num-workers 2

Repo root: /content/outlierdrive/eomt
Config: /content/outlierdrive/eomt/configs/dinov2/cityscapes/semantic/eomt_base_640.yaml
Checkpoint: /content/drive/MyDrive/eomt_finetuning/head_upscale_q/epoch=epoch=9-step=step=1850.ckpt
Data path: /content/drive/MyDrive/cityscapes
Device: cuda:0
Image size: (640, 640)
num_q: 200
Data module: <class 'datasets.cityscapes_semantic.CityscapesSemantic'>
Data img_size: (640, 640)
Data num_classes: 19
2026-06-05 09:10:30.080954: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-06-05 09:10:30.143840: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other ope

## Whole Head+attention of 10 and 11 blocks fine-tuning with smaller Learning Rate.

In [ ]:
!python main.py fit \
    -c configs/dinov2/cityscapes/semantic/eomt_base_640.yaml \
    --data.path /content/drive/MyDrive/cityscapes \
    --model.ckpt_path /content/eomt_coco.bin \
    --model.load_ckpt_class_head False \
    --model.network.num_q 200 \
    --data.img_size "[640,640]" \
    --trainer.devices 1 \
    --trainer.max_epochs 10 \
    --trainer.limit_val_batches 0 \
    --data.batch_size 16 \
    --data.num_workers 2 \
    --trainer.callbacks+=lightning.pytorch.callbacks.ModelCheckpoint \
    --trainer.callbacks.dirpath /content/drive/MyDrive/eomt_finetuning/whole_head_atten1011 \
    --trainer.callbacks.filename "epoch={epoch}-step={step}" \
    --trainer.callbacks.save_top_k -1 \
    --trainer.callbacks.every_n_epochs 1 \
    --trainer.callbacks.save_last True

2026-06-05 10:34:28.704507: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-06-05 10:34:28.768677: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
Seed set to 0
Trainable parameters:
encoder.backbone.blocks.10.attn.qkv.weight
encoder.backbone.blocks.10.attn.qkv.bias
encoder.backbone.blocks.10.attn.proj.weight
encoder.backbone.blocks.10.attn.proj.bias
encoder.backbone.blocks.11.attn.qkv.weight
encoder.backbone.blocks.11.attn.qkv.bias
encoder.backbone.blocks.11.attn.proj.weight
encoder.backbone.blocks.11.attn.proj.bia

Evaluate whole head + 10 and 11 blocks
## 5th epoch

In [ ]:
!python /content/outlierdrive/step4_eomt_eval/eomt_eval_iou.py \
  --repo-root /content/outlierdrive/eomt \
  --config /content/outlierdrive/eomt/configs/dinov2/cityscapes/semantic/eomt_base_640.yaml \
  --checkpoint /content/drive/MyDrive/eomt_finetuning/whole_head_atten1011/epoch=epoch=4-step=step=925.ckpt \
  --data-path /content/drive/MyDrive/cityscapes \
  --output-dir /content/drive/MyDrive/eomt_valset_predictions/coco_finetuned_wholehead1011_5e \
  --device cuda:0 \
  --img-size 640 640 \
  --num-q 200 \
  --num-workers 2

Repo root: /content/outlierdrive/eomt
Config: /content/outlierdrive/eomt/configs/dinov2/cityscapes/semantic/eomt_base_640.yaml
Checkpoint: /content/drive/MyDrive/eomt_finetuning/whole_head_atten1011/epoch=epoch=4-step=step=925.ckpt
Data path: /content/drive/MyDrive/cityscapes
Device: cuda:0
Image size: (640, 640)
num_q: 200
Data module: <class 'datasets.cityscapes_semantic.CityscapesSemantic'>
Data img_size: (640, 640)
Data num_classes: 19
model.safetensors: 100% 346M/346M [00:03<00:00, 98.4MB/s]
2026-06-06 07:46:25.522991: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
Trainable parameters:
encoder.backbone.blocks.10.attn.qkv.weight
encoder.backbone.blocks.10.attn.qkv.bias
encoder.backbone.blocks.10.attn.proj.weight
encoder.backbone.blocks.10.a

## 8 epoch

In [15]:
!python /content/outlierdrive/step4_eomt_eval/eomt_eval_iou.py \
  --repo-root /content/outlierdrive/eomt \
  --config /content/outlierdrive/eomt/configs/dinov2/cityscapes/semantic/eomt_base_640.yaml \
  --checkpoint /content/drive/MyDrive/eomt_finetuning/whole_head_atten1011/epoch=epoch=7-step=step=1480.ckpt \
  --data-path /content/drive/MyDrive/cityscapes \
  --output-dir /content/drive/MyDrive/eomt_valset_predictions/coco_finetuned_wholehead1011_8e \
  --device cuda:0 \
  --img-size 640 640 \
  --num-q 200 \
  --num-workers 2

Repo root: /content/outlierdrive/eomt
Config: /content/outlierdrive/eomt/configs/dinov2/cityscapes/semantic/eomt_base_640.yaml
Checkpoint: /content/drive/MyDrive/eomt_finetuning/whole_head_atten1011/epoch=epoch=7-step=step=1480.ckpt
Data path: /content/drive/MyDrive/cityscapes
Device: cuda:0
Image size: (640, 640)
num_q: 200
Data module: <class 'datasets.cityscapes_semantic.CityscapesSemantic'>
Data img_size: (640, 640)
Data num_classes: 19
2026-06-09 10:04:37.376301: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
Trainable parameters:
encoder.backbone.blocks.10.attn.qkv.weight
encoder.backbone.blocks.10.attn.qkv.bias
encoder.backbone.blocks.10.attn.proj.weight
encoder.backbone.blocks.10.attn.proj.bias
encoder.backbone.blocks.11.attn.qkv.weight


## 10 epochs

In [ ]:
!python /content/outlierdrive/step4_eomt_eval/eomt_eval_iou.py \
  --repo-root /content/outlierdrive/eomt \
  --config /content/outlierdrive/eomt/configs/dinov2/cityscapes/semantic/eomt_base_640.yaml \
  --checkpoint /content/drive/MyDrive/eomt_finetuning/whole_head_atten1011/epoch=epoch=9-step=step=1850.ckpt \
  --data-path /content/drive/MyDrive/cityscapes \
  --output-dir /content/drive/MyDrive/eomt_valset_predictions/coco_finetuned_wholehead1011_10e \
  --device cuda:0 \
  --img-size 640 640 \
  --num-q 200 \
  --num-workers 2

Repo root: /content/outlierdrive/eomt
Config: /content/outlierdrive/eomt/configs/dinov2/cityscapes/semantic/eomt_base_640.yaml
Checkpoint: /content/drive/MyDrive/eomt_finetuning/whole_head_atten1011/epoch=epoch=9-step=step=1850.ckpt
Data path: /content/drive/MyDrive/cityscapes
Device: cuda:0
Image size: (640, 640)
num_q: 200
Data module: <class 'datasets.cityscapes_semantic.CityscapesSemantic'>
Data img_size: (640, 640)
Data num_classes: 19
2026-06-06 07:54:13.199063: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
Trainable parameters:
encoder.backbone.blocks.10.attn.qkv.weight
encoder.backbone.blocks.10.attn.qkv.bias
encoder.backbone.blocks.10.attn.proj.weight
encoder.backbone.blocks.10.attn.proj.bias
encoder.backbone.blocks.11.attn.qkv.weight


---

## Conclusion

Using the above-mentioned setup the best possible improvement we could reach was 69% of mIoU which is pretty decent result taking into account the resource constrainst and the fact that only Learning Rate was adjusted a bit for training the attention layer.

